# 06: Short-Term Forecasting (Geopolitical Context)

This notebook focuses on **short-term fuel price forecasting** using a smaller, recent time window.
It is useful when market dynamics shift quickly due to geopolitical events (for example, Middle East tensions).

## Important note
This model is still statistical and does not directly ingest live geopolitical signals. Use forecasts as a baseline and combine with scenario analysis.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

plt.style.use('seaborn-v0_8-whitegrid')

## 6.1 Load audited data

In [ ]:
audited_dir = os.path.join('..', 'data', 'audited')
source1 = pd.read_csv(os.path.join(audited_dir, 'source1_audited.csv'))
source1['date'] = pd.to_datetime(source1['date'], errors='coerce')

# Prefer USD-standardized columns; fall back safely if legacy GBP-only audited data exists.
if 'crude_oil_price_usd' in source1.columns:
    source1['crude_oil_price_usd'] = pd.to_numeric(source1['crude_oil_price_usd'], errors='coerce')
elif 'crude_oil_price' in source1.columns:
    source1['crude_oil_price_usd'] = pd.to_numeric(source1['crude_oil_price'], errors='coerce')
elif 'crude_oil_price_gbp' in source1.columns:
    source1['crude_oil_price_usd'] = pd.to_numeric(source1['crude_oil_price_gbp'], errors='coerce') * 1.25
else:
    raise KeyError('No crude oil price column found in source1_audited.csv')

ts_full = source1[['date', 'crude_oil_price_usd']].dropna().sort_values('date').set_index('date')['crude_oil_price_usd']
ts_full = ts_full.asfreq('MS')

print('Full series range:', ts_full.index.min(), 'to', ts_full.index.max())
print('Total observations:', len(ts_full))
print('Modeling currency: USD per barrel')

Full series range: 1970-01-01 00:00:00 to 2026-03-01 00:00:00
Total observations: 675


## 6.2 Define short-term window
Use a smaller, recent history to better reflect current market regime.

In [3]:
lookback_months = 60   # Try 36, 48, or 60
forecast_horizon = 6   # 3 to 6 months recommended for short-term
test_horizon = 6

ts_short = ts_full.tail(lookback_months).dropna()
train = ts_short.iloc[:-test_horizon]
test = ts_short.iloc[-test_horizon:]

print('Short window range:', ts_short.index.min(), 'to', ts_short.index.max())
print('Train size:', len(train), 'Test size:', len(test))

Short window range: 2021-04-01 00:00:00 to 2026-03-01 00:00:00
Train size: 54 Test size: 6


## 6.3 Train and compare short-term models
Compare ARIMA vs SARIMA on the recent window only.

In [7]:
arima_short = ARIMA(train, order=(1, 1, 1)).fit()
sarima_short = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 0, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

arima_pred = arima_short.forecast(steps=test_horizon)
sarima_pred = sarima_short.forecast(steps=test_horizon)

results = pd.DataFrame({
    'Model': ['ARIMA_short', 'SARIMA_short'],
    'RMSE': [
        np.sqrt(mean_squared_error(test, arima_pred)),
        np.sqrt(mean_squared_error(test, sarima_pred))
    ],
    'MAPE': [
        mean_absolute_percentage_error(test, arima_pred),
        mean_absolute_percentage_error(test, sarima_pred)
    ]
}).sort_values('RMSE')

results

,Model,RMSE,MAPE
1,SARIMA_short,8.299709,0.089680
0,ARIMA_short,8.421693,0.091687


## 6.4 Refit best short-term model and forecast next months

In [8]:
best_model_name = results.iloc[0]['Model']

if best_model_name == 'SARIMA_short':
    best_model = SARIMAX(
        ts_short,
        order=(1, 1, 1),
        seasonal_order=(1, 0, 1, 12),
        enforce_stationarity=False,
        enforce_invertibility=False
    ).fit(disp=False)
    forecast_obj = best_model.get_forecast(steps=forecast_horizon)
    fc_values = forecast_obj.predicted_mean
    fc_ci = forecast_obj.conf_int(alpha=0.2)  # 80% interval
else:
    best_model = ARIMA(ts_short, order=(1, 1, 1)).fit()
    forecast_obj = best_model.get_forecast(steps=forecast_horizon)
    fc_values = forecast_obj.predicted_mean
    fc_ci = forecast_obj.conf_int(alpha=0.2)

fc_df = pd.DataFrame({
    'forecast': fc_values,
    'lower_80': fc_ci.iloc[:, 0],
    'upper_80': fc_ci.iloc[:, 1]
})

print('Best short-term model:', best_model_name)
fc_df

Best short-term model: SARIMA_short


,forecast,lower_80,upper_80
2026-04-01,76.429268,70.654341,82.204195
2026-05-01,73.600131,63.998764,83.201499
2026-06-01,74.584079,62.886128,86.282029
2026-07-01,73.852483,60.117659,87.587308
2026-08-01,74.480878,59.097212,89.864544
2026-09-01,74.297938,57.367870,91.228006


In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=ts_short.index, y=ts_short.values, mode='lines', name='Recent history', line=dict(color='black')
))
fig.add_trace(go.Scatter(
    x=fc_df.index, y=fc_df['forecast'], mode='lines+markers', name='Short-term forecast', line=dict(color='crimson')
))
fig.add_trace(go.Scatter(
    x=fc_df.index, y=fc_df['upper_80'], mode='lines', line=dict(width=0), showlegend=False
))
fig.add_trace(go.Scatter(
    x=fc_df.index, y=fc_df['lower_80'], mode='lines', line=dict(width=0), fill='tonexty',
    fillcolor='rgba(220,20,60,0.18)', name='80% uncertainty band'
))

fig.update_layout(
    title='Short-Term Fuel Price Forecast (Recent Window)',
    xaxis_title='Date',
    yaxis_title='Crude Oil Price (USD per barrel)',
    template='plotly_white'
)
fig.show()

## 6.5 Interpretation for current tensions

- This forecast is tuned to **recent behavior** and is better for near-term planning.
- The uncertainty band widens as horizon increases, especially in volatile periods.
- For geopolitical shocks, we could consider updating weekly/monthly and combining with scenario overlays (base/upside/downside).

## 6.6 Next Step: Real-Time Monitoring Notebook

Real-time market pull is available in `08_realtime_monitoring.ipynb` as this notebook is focused on reproducible short-term modeling.

Open the new notebook to run live API ingestion and monitoring checks.